# Bài học 11: Nối chuỗi Agent (Agent Chaining)

## Agent Chaining là gì?

**Agent chaining** là khi đầu ra của **agent này** trở thành đầu vào cho **agent tiếp theo**.

Hãy hình dung như một **dây chuyền lắp ráp** trong nhà máy:
- Công nhân 1 làm bước 1, chuyển sang công nhân 2
- Công nhân 2 làm bước 2, chuyển sang công nhân 3
- ...

Trong AI:
- Researcher Agent **tìm kiếm thông tin** rồi chuyển cho Writer Agent
- Writer Agent **viết bài** từ thông tin đó rồi chuyển cho Editor Agent
- ...

Mỗi agent chỉ **làm tốt một việc duy nhất**, rồi bàn giao cho agent kế tiếp.

## Kế hoạch: 2 Agent nối chuỗi

Chúng ta sẽ xây dựng:

```
[Agent 1: Researcher]  →  ghi chú nghiên cứu  →  [Agent 2: Writer]
    Tìm kiếm trên web                              Viết bài từ nghiên cứu
```

- **Agent 1 (Researcher):** Có tool DuckDuckGo, tìm kiếm trên web
- **Agent 2 (Writer):** Nhận kết quả nghiên cứu, viết thành đoạn văn

Cách nối chuỗi rất đơn giản:
```python
research = researcher.run("Search for info...")
article = writer.run(f"Write from: {research.content}")  # Truyền đầu ra làm đầu vào!
```

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from agno.agent import Agent
from agno.models.anthropic import Claude
from agno.tools.duckduckgo import DuckDuckGoTools

# Agent 1: Researcher — tìm kiếm trên web
researcher = Agent(
    name="Researcher",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    tools=[DuckDuckGoTools()],
    instructions=["Search the web and return detailed research notes."],
)

# Agent 2: Writer — viết bài từ nghiên cứu
writer = Agent(
    name="Writer",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    instructions=[
        "Write a short paragraph (3-4 sentences) based on the research notes provided.",
        "Write clearly and professionally.",
    ],
)

print("Đã tạo 2 agent: Researcher và Writer")

In [ ]:
# Bước 1: Nghiên cứu
print("Bước 1: Đang nghiên cứu...")
research = researcher.run("Find information about AI trends in SEO for 2026")
print(f"Nghiên cứu xong! ({len(research.content)} ký tự)\n")

# Bước 2: Viết bài từ nghiên cứu
print("Bước 2: Đang viết bài...")
article = writer.run(f"Write a paragraph from these research notes:\n\n{research.content}")
print(f"Bài viết:\n{article.content}")

## Vừa xảy ra chuyện gì?

Hãy theo dõi luồng xử lý:

1. **Researcher** nhận câu hỏi, tìm kiếm trên DuckDuckGo, và trả về `research.content`
2. `research.content` được **truyền vào** prompt của Writer
3. **Writer** nhận kết quả nghiên cứu và viết thành đoạn văn hoàn chỉnh

**Dòng code then chốt:**
```python
article = writer.run(f"Write a paragraph from these research notes:\n\n{research.content}")
```

Chỉ cần truyền `research.content` (đầu ra của agent trước) vào `writer.run()` — vậy là xong!

## Đây chính xác là cách Pipeline thực tế của chúng ta hoạt động!

Dự án `agentic-content-seo` của chúng ta sử dụng **đúng nguyên lý này** để tạo bài viết SEO tự động:

1. **Research Agent** — Tìm kiếm xu hướng, từ khóa, nội dung đối thủ
2. **Outline Agent** — Tạo cấu trúc bài viết từ nghiên cứu
3. **Writer Agent** — Viết toàn bộ bài viết từ dàn ý
4. **Image Agent** — Tìm và chèn hình ảnh vào bài viết

Mỗi bước truyền dữ liệu cho bước tiếp theo — đúng y như những gì bạn vừa xây dựng!

In [ ]:
# Sơ đồ pipeline thực tế của chúng ta
print("""
Pipeline của chúng ta:

  [Research Agent] --nghiên cứu--> [Outline Agent] --dàn ý--> [Writer Agent] --bài viết--> [Image Agent]
       |                                |                           |                           |
   Tìm kiếm web                  Cấu trúc nội dung           Viết toàn bộ bài            Tìm hình ảnh
""")

## Tiến độ Bài học 11 & Mô-đun 3

### Bài học 11 — Nối chuỗi Agent:
- **Agent chaining**: đầu ra của agent này trở thành đầu vào cho agent tiếp theo
- Cách thực hiện: truyền `response.content` vào `agent.run()`
- Mỗi agent chỉ cần **làm tốt một nhiệm vụ duy nhất**

### Mô-đun 3 đến giờ — Xây dựng Agent:

| Bài học | Khái niệm |
|---------|----------|
| Bài học 8 | Tạo agent cơ bản với `name`, `model`, `instructions` |
| Bài học 9 | Thêm **tools** để agent có thể hành động (tìm kiếm web) |
| Bài học 10 | **Structured output** — bắt agent trả về dữ liệu có cấu trúc |
| Bài học 11 | **Nối chuỗi agent** — xây dựng pipeline tự động |

**Bài học tiếp theo (12):** Chúng ta sẽ xây dựng một **mini pipeline 3 agent** phức tạp hơn với schema lồng nhau — làm cầu nối từ những ví dụ đơn giản này đến code production mà bạn sẽ thấy ở Mô-đun 4.

## Bài tập

Thêm một **agent thứ ba** vào chuỗi: một Editor để cải thiện đầu ra của Writer.

1. Tạo một agent `editor` với instructions kiểu: "You are a content editor. Improve the given text: fix grammar, make it more engaging, and ensure it's SEO-friendly. Return only the improved text."
2. Nối chuỗi: truyền `article.content` vào `editor.run()`
3. In ra cả phiên bản gốc lẫn phiên bản đã chỉnh sửa

Đây chính xác là pattern mà pipeline thực tế của chúng ta sử dụng — chỉ là với nhiều agent hơn!

In [ ]:
# Bài tập: Viết code của bạn ở đây
